In [2]:
!pip install openpyxl scikit-learn

In [3]:
import pandas as pd

from sklearn.tree import DecisionTreeClassifier

from sklearn.preprocessing import LabelEncoder

In [4]:
df = pd.read_excel("CloudGuardian_Normalized_Findings.xlsx")

df

FileNotFoundError: [Errno 2] No such file or directory: 'CloudGuardian_Normalized_Findings.xlsx'

In [6]:
import os

os.listdir()


['.config', 'CloudGuardian_Normalized_Findings.xlsx', 'drive', 'sample_data']

In [7]:
import pandas as pd

df = pd.read_excel("CloudGuardian_Normalized_Findings.xlsx")

df.head()

,MC_ID,Resource,Category,Misconfiguration,Prowler_Result,ScoutSuite_Result,Steampipe_Result,Overall_Status,Notes
0,MC-01,Storage Account,Storage,Public Network Access Enabled,Detected,Detected,Verified,Verified,NaN
1,MC-02,Storage Account,Storage,Blob Public Access Enabled,Detected,Detected,Verified,Verified,NaN
2,MC-03,Storage Account,Storage,Shared Key Authentication Enabled,Detected,Detected,Verified,Verified,NaN
3,MC-04,Storage Account,Identity,Microsoft Entra (OAuth) Authentication Disabled,Detected,Detected,Verified,Verified,NaN
4,MC-05,SQL Server,Database,Public Network Access Enabled,Detected,Detected,Verified,Verified,NaN


In [8]:
# Add feature columns

df["CVSS"] = [
8.2,
9.1,
7.5,
7.4,
8.8,
8.6,
6.5,
9.8,
7.5,
7.8,
7.4,
5.3,
9.1
]

df["Exposure"] = [
5,
5,
4,
3,
5,
5,
5,
5,
5,
4,
3,
2,
3
]

df["Blast_Radius"] = [
4,
5,
3,
3,
5,
4,
3,
5,
3,
4,
3,
5,
5
]

df.head()

,MC_ID,Resource,Category,Misconfiguration,Prowler_Result,ScoutSuite_Result,Steampipe_Result,Overall_Status,Notes,CVSS,Exposure,Blast_Radius
0,MC-01,Storage Account,Storage,Public Network Access Enabled,Detected,Detected,Verified,Verified,NaN,8.2,5,4
1,MC-02,Storage Account,Storage,Blob Public Access Enabled,Detected,Detected,Verified,Verified,NaN,9.1,5,5
2,MC-03,Storage Account,Storage,Shared Key Authentication Enabled,Detected,Detected,Verified,Verified,NaN,7.5,4,3
3,MC-04,Storage Account,Identity,Microsoft Entra (OAuth) Authentication Disabled,Detected,Detected,Verified,Verified,NaN,7.4,3,3
4,MC-05,SQL Server,Database,Public Network Access Enabled,Detected,Detected,Verified,Verified,NaN,8.8,5,5


In [9]:
# Calculate Priority Score
df["Priority_Score"] = (
    df["CVSS"] +
    df["Exposure"] +
    df["Blast_Radius"]
)

# Convert score into Risk Level
def classify(score):
    if score >= 17:
        return "Critical"
    elif score >= 13:
        return "High"
    elif score >= 9:
        return "Medium"
    else:
        return "Low"

df["Risk"] = df["Priority_Score"].apply(classify)

# Show only the important columns
df[["MC_ID","CVSS","Exposure","Blast_Radius","Priority_Score","Risk"]]

,MC_ID,CVSS,Exposure,Blast_Radius,Priority_Score,Risk
0,MC-01,8.2,5,4,17.2,Critical
1,MC-02,9.1,5,5,19.1,Critical
2,MC-03,7.5,4,3,14.5,High
3,MC-04,7.4,3,3,13.4,High
4,MC-05,8.8,5,5,18.8,Critical
5,MC-06,8.6,5,4,17.6,Critical
6,MC-07,6.5,5,3,14.5,High
7,MC-08,9.8,5,5,19.8,Critical
8,MC-09,7.5,5,3,15.5,High
9,MC-10,7.8,4,4,15.8,High


In [10]:
from sklearn.preprocessing import LabelEncoder

# Create the encoder
encoder = LabelEncoder()

# Convert Risk labels to numbers
df["Risk_Label"] = encoder.fit_transform(df["Risk"])

# Display the mapping
mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
print("Risk Label Mapping:")
print(mapping)

# View the new column
df[["MC_ID", "Risk", "Risk_Label"]]

Risk Label Mapping:
{'Critical': np.int64(0), 'High': np.int64(1), 'Medium': np.int64(2)}


,MC_ID,Risk,Risk_Label
0,MC-01,Critical,0
1,MC-02,Critical,0
2,MC-03,High,1
3,MC-04,High,1
4,MC-05,Critical,0
5,MC-06,Critical,0
6,MC-07,High,1
7,MC-08,Critical,0
8,MC-09,High,1
9,MC-10,High,1


In [11]:
# Features (Input)
X = df[["CVSS", "Exposure", "Blast_Radius"]]

# Target (Output)
y = df["Risk_Label"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

Features:
   CVSS  Exposure  Blast_Radius
0   8.2         5             4
1   9.1         5             5
2   7.5         4             3
3   7.4         3             3
4   8.8         5             5

Target:
0    0
1    0
2    1
3    1
4    0
Name: Risk_Label, dtype: int64


In [12]:
from sklearn.tree import DecisionTreeClassifier

# Create the model
model = DecisionTreeClassifier(random_state=42)

# Train the model
model.fit(X, y)

print("✅ Decision Tree model trained successfully!")

✅ Decision Tree model trained successfully!


In [13]:
# Predict the risk level
df["Predicted_Label"] = model.predict(X)

# Convert numeric labels back to text
df["Predicted_Risk"] = encoder.inverse_transform(df["Predicted_Label"])

# Display results
df[["MC_ID", "Risk", "Predicted_Risk"]]

,MC_ID,Risk,Predicted_Risk
0,MC-01,Critical,Critical
1,MC-02,Critical,Critical
2,MC-03,High,High
3,MC-04,High,High
4,MC-05,Critical,Critical
5,MC-06,Critical,Critical
6,MC-07,High,High
7,MC-08,Critical,Critical
8,MC-09,High,High
9,MC-10,High,High


In [14]:
from sklearn.model_selection import train_test_split

# Features
X = df[["CVSS", "Exposure", "Blast_Radius"]]

# Target
y = df["Risk_Label"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

print("Training samples :", len(X_train))
print("Testing samples  :", len(X_test))

Training samples : 9
Testing samples  : 4


In [15]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)

model.fit(X_train, y_train)

print("Decision Tree trained successfully!")

Decision Tree trained successfully!


In [16]:
y_pred = model.predict(X_test)

print("Predicted Labels:")
print(y_pred)

Predicted Labels:
[0 0 0 1]


In [17]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.5

Classification Report
              precision    recall  f1-score   support

           0       0.33      1.00      0.50         1
           1       1.00      0.50      0.67         2
           2       0.00      0.00      0.00         1

    accuracy                           0.50         4
   macro avg       0.44      0.50      0.39         4
weighted avg       0.58      0.50      0.46         4


Confusion Matrix
[[1 0 0]
 [1 1 0]
 [1 0 0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [18]:
df.to_excel("Prioritized_Findings.xlsx", index=False)

print("Prioritized findings exported successfully!")

Prioritized findings exported successfully!
